# rds_mspas_guatemala_ingest


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import lit

MSPAS_TABLES = [
    ("raw_data.mspas_mortalidad_general_2010_2024", "sandbox.raw_mspas_mortalidad_general"),
    ("raw_data.mspas_tasa_mortalidad_2001_2019",    "sandbox.raw_mspas_tasa"),
]


def _jdbc_opts(dbtable: str) -> dict:
    DB_USER = dbutils.secrets.get(scope="semis2_scope", key="DB_USER")
    DB_PASS = dbutils.secrets.get(scope="semis2_scope", key="DB_PASS")
    DB_HOST = dbutils.secrets.get(scope="semis2_scope", key="DB_HOST")
    DB_PORT = dbutils.secrets.get(scope="semis2_scope", key="DB_PORT")
    DB_NAME = dbutils.secrets.get(scope="semis2_scope", key="DB_NAME")
    return {
        "url":      f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}",
        "dbtable":  dbtable,
        "user":     DB_USER,
        "password": DB_PASS,
        "driver":   "org.postgresql.Driver",
    }


def test_connection() -> None:
    df_tables = (
        spark.read.format("jdbc")
        .options(**_jdbc_opts("information_schema.tables"))
        .option("dbtable", "information_schema.tables")
        .load()
        .filter(F.col("table_schema") == "raw_data")
        .select("table_schema", "table_name")
    )
    print("Tablas en raw_data:")
    df_tables.show(truncate=False)


# ── Ejecución ──────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS sandbox")
test_connection()

for rds_table, delta_table in MSPAS_TABLES:
    df = (
        spark.read.format("jdbc").options(**_jdbc_opts(rds_table)).load()
        .withColumn("_source", F.lit(rds_table))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )
    display(df)

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(delta_table)
